In [ ]:
# Imports
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO
import os
from pathlib import Path
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [ ]:
# ---- 1) Paths & device
root = Path("Downloads/ImageNet").expanduser()  # your 5-class subset
assert root.exists(), f"Path not found: {root}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Pretrained ResNet-18 and its paired preprocessing
weights = models.ResNet18_Weights.IMAGENET1K_V1
resnet18 = models.resnet18(weights=weights).to(device).eval()
preprocess = weights.transforms()  # <- handles resize/crop/normalize for these weights


In [ ]:

# Dataset & loader (no shuffle so we can refer back to file paths deterministically)
dataset = ImageFolder(str(root), transform=preprocess)
loader = DataLoader(dataset, batch_size=5, shuffle=True, num_workers=0)

# Human-readable ImageNet-1K labels
imagenet_labels = weights.meta["categories"]


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CSAE(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = nn.Conv2d(3, 32, 3, stride=2, padding=1)  # 224 -> 112
        self.enc2 = nn.Conv2d(32, 64, 3, stride=2, padding=1) # 112 -> 56
        self.enc3 = nn.Conv2d(64, 128, 3, stride=2, padding=1)# 56 -> 28

        # Bottleneck (latent z)
        self.enc4 = nn.Conv2d(128, 256, 3, stride=2, padding=1)# 28 -> 14

        # Decoder
        self.dec1 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1) # 14 -> 28
        self.dec2 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)  # 28 -> 56
        self.dec3 = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)   # 56 -> 112
        self.dec4 = nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1)    # 112 -> 224

    def forward(self, x):
        # Encoder
        x = F.relu(self.enc1(x))
        x = F.relu(self.enc2(x))
        x = F.relu(self.enc3(x))
        z = F.relu(self.enc4(x))  # latent

        # Decoder
        x = F.relu(self.dec1(z))
        x = F.relu(self.dec2(x))
        x = F.relu(self.dec3(x))
        logits = self.dec4(x)     # mask logits

        return logits


In [ ]:
# Sanity check CSAE with random input
csae = CSAE()

# Random input: batch of 1 image, 3 channels, 224x224
x = torch.randn(1, 3, 224, 224)

logits = csae(x)

print("Input shape :", x.shape)
print("Output shape:", logits.shape)

Input shape : torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 1, 224, 224])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """(conv => ReLU => conv => ReLU)"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetCSAE(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)   # 224 -> 112
        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)   # 112 -> 56
        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)   # 56 -> 28
        self.down4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)   # 28 -> 14

        # Bottleneck (latent)
        self.bottleneck = DoubleConv(256, 512)

        # Decoder
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2) # 14 -> 28
        self.dec4 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2) # 28 -> 56
        self.dec3 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)  # 56 -> 112
        self.dec2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)   # 112 -> 224
        self.dec1 = DoubleConv(64, 32)

        # Output mask logits
        self.out_conv = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        # Encoder
        d1 = self.down1(x)
        d2 = self.down2(self.pool1(d1))
        d3 = self.down3(self.pool2(d2))
        d4 = self.down4(self.pool3(d3))
        bn = self.bottleneck(self.pool4(d4))

        # Decoder with skip connections
        u4 = self.up4(bn)
        u4 = torch.cat([u4, d4], dim=1)
        u4 = self.dec4(u4)

        u3 = self.up3(u4)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.dec3(u3)

        u2 = self.up2(u3)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.dec2(u2)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.dec1(u1)

        logits = self.out_conv(u1)
        return logits


In [ ]:
model = UNetCSAE()
x = torch.randn(1, 3, 224, 224)
logits = model(x)

print("Input :", x.shape)
print("Output:", logits.shape)


Input : torch.Size([1, 3, 224, 224])
Output: torch.Size([1, 1, 224, 224])


In [ ]:
from torchvision import models

resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
print(resnet18)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
# Dict to store activations
acts = {}

def get_hook(name):
    def hook(module, input, output):
        acts[name] = output.detach()
    return hook

# Register hooks
hooks = []
hooks.append(resnet18.relu.register_forward_hook(get_hook("relu")))
hooks.append(resnet18.layer1.register_forward_hook(get_hook("layer1")))
hooks.append(resnet18.layer2.register_forward_hook(get_hook("layer2")))
hooks.append(resnet18.layer3.register_forward_hook(get_hook("layer3")))
hooks.append(resnet18.layer4.register_forward_hook(get_hook("layer4")))
hooks.append(resnet18.avgpool.register_forward_hook(get_hook("avgpool")))


In [ ]:
# Grab 1 batch (say 1 image for simplicity)
images, targets = next(iter(loader))
images = images[:1]

# Run through model
with torch.no_grad():
    _ = resnet18(images)

# Print captured activations
for k, v in acts.items():
    print(f"{k:7s} | shape = {tuple(v.shape)}")


relu    | shape = (1, 64, 112, 112)
layer1  | shape = (1, 64, 56, 56)
layer2  | shape = (1, 128, 28, 28)
layer3  | shape = (1, 256, 14, 14)
layer4  | shape = (1, 512, 7, 7)
avgpool | shape = (1, 512, 1, 1)


In [ ]:
# -------------------------------
# 1. Cross-Entropy Loss
# -------------------------------
def ce_loss(logits_x, logits_e):
    """
    Cross-entropy loss between explanation logits and
    the top-1 class predicted by the original image.
    """
    y_star = logits_x.argmax(dim=1)
    return F.cross_entropy(logits_e, y_star)


# -------------------------------
# 2. KL Divergence Loss
# -------------------------------
def kl_loss(logits_x, logits_e, temperature=1.0):
    """
    KL divergence between softmax distributions of
    original and explanation logits.
    """
    p_x = F.softmax(logits_x / temperature, dim=1)
    log_p_e = F.log_softmax(logits_e / temperature, dim=1)
    return F.kl_div(log_p_e, p_x, reduction="batchmean") * (temperature ** 2)


# -------------------------------
# 3a. Cosine Similarity Loss
# -------------------------------
def cosine_loss(feat_x, feat_e, eps=1e-8):
    """
    Cosine similarity loss between feature maps.
    Supports both 2D vectors and 4D feature maps.
    """
    # Flatten (batch, C, H, W) -> (batch, C, H*W)
    if feat_x.dim() == 4:
        B, C, H, W = feat_x.shape
        feat_x = feat_x.view(B, C, -1)
        feat_e = feat_e.view(B, C, -1)

    # Normalize
    feat_x = F.normalize(feat_x, dim=1, eps=eps)
    feat_e = F.normalize(feat_e, dim=1, eps=eps)

    # Cosine similarity per location
    cos = (feat_x * feat_e).sum(dim=1)  # (B, H*W)
    loss = 1 - cos.mean()               # average over batch & spatial
    return loss


# -------------------------------
# 3b. MSE Loss (with normalization)
# -------------------------------
def mse_loss(feat_x, feat_e):
    """
    Mean squared error between normalized feature maps.
    """
    # Normalize features across channels
    def norm(t):
        B, C, *spatial = t.shape
        flat = t.view(B, C, -1)
        mean = flat.mean(dim=-1, keepdim=True)
        std = flat.std(dim=-1, keepdim=True) + 1e-6
        normed = (flat - mean) / std
        return normed.view(B, C, *spatial)

    feat_x = norm(feat_x)
    feat_e = norm(feat_e)
    return F.mse_loss(feat_x, feat_e)

# -------------------------------
# 4. Sparsity / Minimality Losses
# -------------------------------

def area_loss(mask_prob):
    """
    L1 area penalty: encourages fewer active pixels.
    mask_prob: (B,1,H,W) with values in [0,1]
    """
    return mask_prob.mean()


def binarization_loss(mask_prob):
    """
    Penalize values between 0 and 1.
    p(1-p) is maximal at 0.5, minimal at 0/1.
    """
    return (mask_prob * (1 - mask_prob)).mean()


def tv_loss(mask_prob):
    """
    Total Variation loss for spatial smoothness.
    Encourages compact, blob-like masks without speckle.
    """
    dh = torch.abs(mask_prob[:, :, 1:, :] - mask_prob[:, :, :-1, :]).mean()
    dw = torch.abs(mask_prob[:, :, :, 1:] - mask_prob[:, :, :, :-1]).mean()
    return dh + dw

# -------------------------------
# 5. Robustness / Abductive Losses
# -------------------------------

def abductive_loss(model, x, mask_prob, baseline, backgrounds, temperature=1.0):
    """
    Ensures outside-mask regions don't matter.
    Replace outside pixels with random backgrounds r,
    require prediction distribution to remain stable.

    Args:
        model        : frozen classifier (e.g. resnet18)
        x            : original image tensor (B,3,H,W)
        mask_prob    : (B,1,H,W) explanation mask (soft or hard)
        baseline     : baseline image tensor (e.g. zeros/gray), shape (B,3,H,W)
        backgrounds  : list of background tensors (same shape as x)
        temperature  : temperature for KL
    """
    B, _, H, W = x.shape
    mask = (mask_prob > 0.5).float()  # hard mask for robustness
    mask = mask.expand_as(x)

    with torch.no_grad():
        logits_x = model(x)
        p_x = F.softmax(logits_x / temperature, dim=1)

    loss = 0.0
    for r in backgrounds:
        r = r.to(x.device)
        # Explanation with random background outside mask
        e_r = mask * x + (1 - mask) * r
        logits_e = model(e_r)
        log_p_e = F.log_softmax(logits_e / temperature, dim=1)
        loss += F.kl_div(log_p_e, p_x, reduction="batchmean") * (temperature ** 2)

    return loss / len(backgrounds)


def leakage_loss(feat_e, mask_prob):
    """
    Prevent leakage of features outside mask.
    Downsample mask to feature map resolution,
    then penalize activations outside the mask.
    """
    # Resize mask to feature map size
    mask_ds = F.interpolate(mask_prob, size=feat_e.shape[-2:], mode="bilinear", align_corners=False)
    return ((1 - mask_ds) * feat_e.abs()).mean()


def necessity_loss(model, x, mask_prob, baseline, margin=1.0):
    """
    Ensures outside-mask alone is insufficient.
    Replace mask region with baseline, require
    target class logit to drop at least by 'margin'.
    """
    mask = (mask_prob > 0.5).float().expand_as(x)
    x_comp = (1 - mask) * x + mask * baseline  # complement image

    with torch.no_grad():
        logits_x = model(x)
        y_star = logits_x.argmax(dim=1)

    logits_comp = model(x_comp)
    # hinge: encourage drop in target logit
    diff = logits_x[range(len(y_star)), y_star] - logits_comp[range(len(y_star)), y_star]
    return torch.clamp(margin - diff, min=0).mean()


In [ ]:

def abductive_loss(model, x, mask_prob, baseline, backgrounds, temperature=1.0):
    mask = (mask_prob > 0.5).float().expand_as(x)
    with torch.no_grad():
        px = F.softmax(model(x) / temperature, dim=1)
    loss = 0.0
    for r in backgrounds:
        r = r.to(x.device)
        e_r = mask * x + (1 - mask) * r
        log_pe = F.log_softmax(model(e_r) / temperature, dim=1)
        loss += F.kl_div(log_pe, px, reduction="batchmean") * (temperature ** 2)
    return loss / max(1, len(backgrounds))

def leakage_loss(feat_e, mask_prob):
    m = F.interpolate(mask_prob, size=feat_e.shape[-2:], mode="bilinear", align_corners=False)
    return ((1 - m) * feat_e.abs()).mean()

def necessity_loss(model, x, mask_prob, baseline, margin=1.0):
    m = (mask_prob > 0.5).float().expand_as(x)
    x_comp = (1 - m) * x + m * baseline
    with torch.no_grad():
        logits_x = model(x)
        y_star = logits_x.argmax(dim=1)
    logits_c = model(x_comp)
    drop = logits_x[range(len(y_star)), y_star] - logits_c[range(len(y_star)), y_star]
    return torch.clamp(margin - drop, min=0).mean()


In [ ]:
import torch
import torch.nn.functional as F

def tv8_loss(p, w_ax=1.0, w_diag=1.0/1.41421356237, eps=0.0):
    # p: (B,1,H,W)
    def diff(a, b):  # L1 (or smooth-L1 if you like)
        d = (a - b).abs()
        return d if eps == 0 else torch.sqrt(d*d + eps)
    # axial shifts
    up    = p[:, :, :-1, :]
    down  = p[:, :, 1:,  :]
    left  = p[:, :, :, :-1]
    right = p[:, :, :, 1:]
    tv = 0.0
    tv += w_ax * diff(down, up).mean()
    tv += w_ax * diff(right, left).mean()
    # diagonal shifts (crop to common overlap)
    center = p[:, :, 1:, 1:]
    up_l   = p[:, :, :-1, :-1]
    up_r   = p[:, :, :-1, 1:]
    down_l = p[:, :, 1:,  :-1]
    down_r = p[:, :, 1:,  1:]
    tv += w_diag * diff(center, up_l).mean()
    tv += w_diag * diff(down_l, up_r).mean()
    return tv


In [ ]:
# ===== full activation-matching training cell (single image, all losses) =====
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision.transforms as T

# --- pick mask net
mask_net = UNetCSAE().to(device) if 'UNetCSAE' in globals() else CSAE().to(device)
mask_net.train()

# --- grab one image
x_batch, y_batch = next(iter(loader))
x = x_batch[:1].to(device)   # (1,3,H,W)
y_folder = y_batch[:1]

# --- baseline & denorm (for display only)
baseline = torch.zeros_like(x)
mean = torch.tensor([0.485, 0.456, 0.406], device=x.device)[:, None, None]
std  = torch.tensor([0.229, 0.224, 0.225], device=x.device)[:, None, None]
def denorm(t):  # (1,3,H,W) -> (H,W,3)
    return torch.clamp((t[0] * std + mean).permute(1,2,0).detach().cpu(), 0, 1)

# --- straight-through binarizer
def hard_st(p, thresh=0.5):
    m_hard = (p > thresh).float()
    return p + (m_hard - p).detach()

# --- run model & capture activations fresh
def run_with_acts(model, x, require_grad: bool):
    model = model.to(x.device).eval()
    acts = {}
    hs = []
    def hook(name):
        def fn(_, __, out): acts[name] = out if require_grad else out.detach()
        return fn
    hs += [model.relu.register_forward_hook(hook("relu")),
           model.layer1.register_forward_hook(hook("layer1")),
           model.layer2.register_forward_hook(hook("layer2")),
           model.layer3.register_forward_hook(hook("layer3")),
           model.layer4.register_forward_hook(hook("layer4")),
           model.avgpool.register_forward_hook(hook("avgpool"))]
    with torch.set_grad_enabled(require_grad):
        logits = model(x)
    for h in hs: h.remove()
    return acts, logits

# --- simple backgrounds (normalized space)
def sample_backgrounds(x, k=3):
    bgs = [
        torch.zeros_like(x),                 # mean gray (in normalized space)
        torch.randn_like(x),                 # Gaussian noise
        torch.empty_like(x).uniform_(-1, 1)  # uniform noise
    ]
    return bgs[:k]

# --- layer-wise activation losses (cosine shallow, mse deep)
layer_weights = {"relu":1.0, "layer1":2.0, "layer2":2.0, "layer3":4.0, "layer4":8.0}
def layer_loss(name, fx, fe):
    return cosine_loss(fx, fe) if name in ["relu","layer1"] else mse_loss(fx, fe)

# --- weights
lam_act = 10.0
lam_ce  = 15.0
lam_kl  = 5.0
lam_area = 100.0
lam_bin  = 1.0
lam_tv   = 500.0
lam_abd  = 100.0
lam_leak = 0.0
lam_nec  = 0.0
nec_margin = 1.0
temperature = 2.0

# --- optimizer
opt = torch.optim.Adam(mask_net.parameters(), lr=5e-5)

# --- train
steps = 880
for t in range(1, steps+1):
    opt.zero_grad()

    # mask & explanation
    mask_logits = mask_net(x)                  # (1,1,H,W)
    mask_prob   = torch.sigmoid(mask_logits)   # (1,1,H,W)
    mask_hard   = hard_st(mask_prob)           # (1,1,H,W), ST

    m3 = mask_hard.expand_as(x)                # (1,3,H,W)
    e  = m3 * x                                # baseline is zeros for now

    # features/logits
    acts_x, logits_x = run_with_acts(resnet18, x, require_grad=False)
    acts_e, logits_e = run_with_acts(resnet18, e, require_grad=True)

    # output-level
    loss_ce_ = ce_loss(logits_x, logits_e)
    loss_kl_ = kl_loss(logits_x, logits_e, temperature=temperature)

    # activation matching
    loss_act_sum, per_layer = 0.0, {}
    for name, w in layer_weights.items():
        L = layer_loss(name, acts_x[name], acts_e[name])
        per_layer[name] = L.item()
        loss_act_sum = loss_act_sum + w * L

    # sparsity/regularity
    loss_area_ = area_loss(mask_prob)
    loss_bin_  = binarization_loss(mask_prob)
    loss_tv_   = tv8_loss(mask_prob)

    # robustness family
    bgs = sample_backgrounds(x, k=3)
    loss_abd_  = abductive_loss(resnet18, x, mask_prob, baseline, bgs, temperature=temperature)
    loss_leak_ = sum(leakage_loss(acts_e[n], mask_prob) for n in ["layer2","layer3","layer4"])
    loss_nec_  = necessity_loss(resnet18, x, mask_prob, baseline, margin=nec_margin)

    # total
    loss_total = (
        lam_act  * loss_act_sum +
        lam_ce   * loss_ce_ +
        lam_kl   * loss_kl_ +
        lam_area * loss_area_ +
        lam_bin  * loss_bin_ +
        lam_tv   * loss_tv_ +
        lam_abd  * loss_abd_ +
        lam_leak * loss_leak_ +
        lam_nec  * loss_nec_
    )

    loss_total.backward()
    opt.step()

    # viz/log
    if t % 10 == 0 or t == 1:
        with torch.no_grad():
            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(denorm(x));  axes[0].set_title("Original"); axes[0].axis("off")
            axes[1].imshow(mask_hard[0,0].cpu(), cmap="gray", vmin=0, vmax=1); axes[1].set_title("Binarized Mask"); axes[1].axis("off")
            axes[2].imshow(denorm(e));  axes[2].set_title("Explanation"); axes[2].axis("off")
            plt.suptitle(f"Step {t}"); plt.tight_layout(); plt.show()

            px = F.softmax(logits_x, dim=1); pe = F.softmax(logits_e, dim=1)
            topx, tope = px.argmax(dim=1).item(), pe.argmax(dim=1).item()
            probx, probe = px[0, topx].item(), pe[0, tope].item()
            print(f"[Step {t}] Top-1(x)={imagenet_labels[topx]} ({probx:.2f}) | Top-1(e)={imagenet_labels[tope]} ({probe:.2f})")

        print(f"  total={loss_total.item():.4f} | act={loss_act_sum.item():.4f} | ce={loss_ce_.item():.4f} | kl={loss_kl_.item():.4f} | "
              f"area={loss_area_.item():.4f} | bin={loss_bin_.item():.4f} | tv={loss_tv_.item():.6f}")
        print(f"  abd={loss_abd_.item():.4f} | leak={loss_leak_.item():.4f} | nec={loss_nec_.item():.4f}")
        print("  per-layer act: " + ", ".join([f"{k}:{per_layer[k]:.3f}" for k in layer_weights.keys()]))



In [ ]:
# ============================================
# Activation graphs (vertical) for x vs e
# Layers: layer0 → layer1 → layer2 → layer3 → layer4 → FC → top-10 classes
# ============================================
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt, networkx as nx

# --------- Assumptions / guards ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert 'resnet18' in globals(), "Please define a pretrained `resnet18` before running."
assert 'x' in globals() and 'e' in globals(), "Expected tensors `x` and `e` from the previous cell."
resnet18 = resnet18.to(device).eval()
x = x.to(device)
e = e.to(device)

# Optional labels fallback
if 'imagenet_labels' not in globals():
    imagenet_labels = [f"class {i}" for i in range(resnet18.fc.out_features)]

# -----------------------
# Forward & capture activations
# -----------------------
@torch.no_grad()
def run_with_acts(inp):
    acts = {}
    # stem to layer0 (conv1+bn1+relu+maxpool)
    z = resnet18.conv1(inp)
    z = resnet18.bn1(z)
    z = resnet18.relu(z)
    z = resnet18.maxpool(z)
    acts["layer0"] = z.clone()

    # residual blocks
    z = resnet18.layer1(z); acts["layer1"] = z.clone()
    z = resnet18.layer2(z); acts["layer2"] = z.clone()
    z = resnet18.layer3(z); acts["layer3"] = z.clone()
    z = resnet18.layer4(z); acts["layer4"] = z.clone()

    z = resnet18.avgpool(z); acts["avgpool"] = z.clone()  # (1,512,1,1)
    f = torch.flatten(z, 1)                               # (1,512)
    logits = resnet18.fc(f)                               # (1,1000)
    return acts, logits

acts_x, logits_x = run_with_acts(x)
acts_e, logits_e = run_with_acts(e)

# -----------------------
# Utilities & static ingress matrices
# -----------------------
def ch_energy(feat):  # (1,C,H,W) -> (C,)
    return torch.sqrt((feat[0]**2).sum(dim=(1,2)) + 1e-12)

def ingress_matrix_for(dst_block):  # conv1 (+downsample if present) → (C_out, C_in)
    mats = [dst_block.conv1.weight.detach().abs().sum(dim=(2,3)).cpu().numpy()]
    if isinstance(getattr(dst_block, "downsample", None), torch.nn.Sequential):
        ds0 = dst_block.downsample[0]
        mats.append(ds0.weight.detach().abs().sum(dim=(2,3)).cpu().numpy())
    return sum(mats) if len(mats) > 1 else mats[0]

INGRESS = {
    ("layer0","layer1"): ingress_matrix_for(resnet18.layer1[0]),
    ("layer1","layer2"): ingress_matrix_for(resnet18.layer2[0]),
    ("layer2","layer3"): ingress_matrix_for(resnet18.layer3[0]),
    ("layer3","layer4"): ingress_matrix_for(resnet18.layer4[0]),
}

# -----------------------
# Selection knobs
# -----------------------
TOP_K = {         # channels per layer to plot
    "layer0": 24,
    "layer1": 24,
    "layer2": 24,
    "layer3": 24,
    "layer4": 32,
    "FC":     48,   # out of 512
}
TOP_EDGES_PER_DST   = 4   # max incoming edges per destination conv channel
TOP_EDGES_TO_CLASS  = 3   # edges from FC into each class node
TOP_CLASSES         = 10  # show top-10 class nodes
EDGE_MIN_FRAC       = 0.0 # relative pruning within each edge-set (0 disables)

# -----------------------
# Choose node IDs using IMAGE x (fixed across both panels)
# -----------------------
idx_x = {}
idx_x["layer0"] = np.argsort(-ch_energy(acts_x["layer0"]).cpu().numpy())[:TOP_K["layer0"]].tolist()
idx_x["layer1"] = np.argsort(-ch_energy(acts_x["layer1"]).cpu().numpy())[:TOP_K["layer1"]].tolist()
idx_x["layer2"] = np.argsort(-ch_energy(acts_x["layer2"]).cpu().numpy())[:TOP_K["layer2"]].tolist()
idx_x["layer3"] = np.argsort(-ch_energy(acts_x["layer3"]).cpu().numpy())[:TOP_K["layer3"]].tolist()
idx_x["layer4"] = np.argsort(-ch_energy(acts_x["layer4"]).cpu().numpy())[:TOP_K["layer4"]].tolist()
idx_x["FC"]     = np.argsort(-acts_x["avgpool"].view(-1).abs().cpu().numpy())[:TOP_K["FC"]].tolist()

# -----------------------
# Build a vertical graph for any input (acts, logits), using idx_x selection
# -----------------------
def build_graph_vertical(acts, logits, idx_sel):
    G = nx.DiGraph()
    # energies for sizing (this input)
    E = {
        "layer0": ch_energy(acts["layer0"]).cpu().numpy(),
        "layer1": ch_energy(acts["layer1"]).cpu().numpy(),
        "layer2": ch_energy(acts["layer2"]).cpu().numpy(),
        "layer3": ch_energy(acts["layer3"]).cpu().numpy(),
        "layer4": ch_energy(acts["layer4"]).cpu().numpy(),
        "FC":     acts["avgpool"].view(-1).abs().cpu().numpy()
    }

    layer_order = ["layer0","layer1","layer2","layer3","layer4","FC","class"]
    y_pos = {k:i for i,k in enumerate(layer_order)}  # 0..6 (we’ll plot as negative y to go top→bottom)
    colors = {"layer0":"#8ECAE6","layer1":"#4C78A8","layer2":"#F58518",
              "layer3":"#54A24B","layer4":"#EECA3B","FC":"#B279A2","class":"#666666"}

    # nodes
    def place_nodes(layer):
        ids = idx_sel.get(layer, [])
        n = max(1, len(ids))
        for r, ch in enumerate(ids):
            xcoord = 0.02 + 0.96 * (r / max(1, n-1))  # spread left→right
            nid = f"{layer}:{ch}"
            size = float(E[layer][ch]) if len(E[layer])>ch else 1.0
            G.add_node(nid, layer=layer, ch=int(ch),
                       pos=(xcoord, -y_pos[layer]),
                       color=colors[layer], size=size)
    for L in ["layer0","layer1","layer2","layer3","layer4","FC"]:
        place_nodes(L)

    # edges between conv stages (tag with layer-pair 'lp')
    for (src, dst), W in INGRESS.items():
        src_ids = idx_sel.get(src, []); dst_ids = idx_sel.get(dst, [])
        if not src_ids or not dst_ids:
            continue
        sub = W[np.ix_(dst_ids, src_ids)]  # (|dst|, |src|)
        # scale by source energy for this input
        src_energy = np.array([E[src][i] for i in src_ids]) + 1e-12
        sub = sub * src_energy[None, :]

        thr = EDGE_MIN_FRAC * (sub.max() + 1e-12) if EDGE_MIN_FRAC>0 and sub.size>0 else 0.0
        lp_id = f"{src}->{dst}"
        for j, dch in enumerate(dst_ids):
            row = sub[j]
            if row.size == 0:
                continue
            keep = np.argsort(-row)[:min(TOP_EDGES_PER_DST, row.size)]
            for k in keep:
                w = float(row[k])
                if w >= thr:
                    sch = src_ids[int(k)]
                    # nodes exist by construction
                    G.add_edge(f"{src}:{sch}", f"{dst}:{dch}", weight=w, lp=lp_id)

    # edges layer4 → FC (avgpool identity; one-to-one for selected channels only)
    layer4_selected = set(idx_sel.get("layer4", []))
    for ch in idx_sel.get("FC", []):
        if ch < len(E["FC"]) and f"FC:{ch}" in G.nodes and (ch in layer4_selected):
            w = float(E["FC"][ch])
            if w > 0:
                G.add_edge(f"layer4:{ch}", f"FC:{ch}", weight=w, lp="layer4->FC")

    # FC → top-10 classes (by this input’s softmax)
    probs = F.softmax(logits, dim=1)[0].detach().cpu().numpy()
    top_idx = np.argsort(-probs)[:TOP_CLASSES]
    fc_w_abs = resnet18.fc.weight.detach().abs().cpu().numpy()  # (1000,512)

    for rank, cls in enumerate(top_idx):
        xcoord = 0.02 + 0.96 * (rank / max(1, len(top_idx)-1))
        nid = f"class:{cls}"
        label_txt = imagenet_labels[cls] if cls < len(imagenet_labels) else f"class {cls}"
        G.add_node(nid, layer="class", ch=int(cls),
                   pos=(xcoord, -y_pos["class"]),
                   color=colors["class"], size=1.0,
                   label=f"{label_txt} ({probs[cls]:.2f})")
        # connect strongest FC contributors among selected indices
        fc_ids = np.array(idx_sel.get("FC", []), int)
        if fc_ids.size > 0:
            contrib = fc_w_abs[cls, fc_ids] * E["FC"][fc_ids]
            thr = EDGE_MIN_FRAC * (contrib.max() + 1e-12) if EDGE_MIN_FRAC>0 and contrib.size>0 else 0.0
            keep = np.argsort(-contrib)[:min(TOP_EDGES_TO_CLASS, contrib.size)]
            for s_idx in keep:
                w = float(contrib[s_idx])
                if w >= thr:
                    ch_src = int(fc_ids[s_idx])
                    if f"FC:{ch_src}" in G.nodes:
                        G.add_edge(f"FC:{ch_src}", nid, weight=w, lp="FC->class")

    return G

Gx = build_graph_vertical(acts_x, logits_x, idx_x)
Ge = build_graph_vertical(acts_e, logits_e, idx_x)

# -----------------------
# Draw (two vertical panels side-by-side)
# -----------------------
def draw_vertical(G, title):
    # only nodes with 'pos'
    nodes_with_pos = [n for n in G.nodes if "pos" in G.nodes[n]]
    pos = {n: G.nodes[n]["pos"] for n in nodes_with_pos}

    # node sizes/colors
    sizes = np.array([G.nodes[n].get("size", 1.0) for n in nodes_with_pos])
    sizes = 120 * (sizes / (sizes.max() + 1e-12)) + 30
    colors = [G.nodes[n].get("color", "#666666") for n in nodes_with_pos]

    # edges with both endpoints present
    edges_draw = [(u, v) for (u, v) in G.edges if u in pos and v in pos]

    # ---- width normalization *within* each layer pair ----
    EDGE_WIDTH_RANGE = (0.6, 3.0)  # min, max line width
    EDGE_GAMMA = 0.8               # <1 boosts contrast; 1 = linear

    from collections import defaultdict
    groups = defaultdict(list)
    for e in edges_draw:
        lp = G.edges[e].get("lp", "UNK")
        groups[lp].append(e)

    widths = []
    for e in edges_draw:
        lp = G.edges[e].get("lp", "UNK")
        grp = groups[lp]
        w_grp = np.array([G.edges[g].get("weight", 1.0) for g in grp], dtype=np.float32)
        w_min, w_max = float(w_grp.min()), float(w_grp.max())
        w_cur = float(G.edges[e].get("weight", 1.0))
        if w_max > w_min:
            w_norm = (w_cur - w_min) / (w_max - w_min)
        else:
            w_norm = 0.5
        if EDGE_GAMMA != 1.0:
            w_norm = w_norm ** EDGE_GAMMA
        w_plot = EDGE_WIDTH_RANGE[0] + (EDGE_WIDTH_RANGE[1] - EDGE_WIDTH_RANGE[0]) * w_norm
        widths.append(w_plot)
    # ------------------------------------------------------

    nx.draw_networkx_nodes(G, pos, nodelist=nodes_with_pos,
                           node_size=sizes.tolist(),
                           node_color=colors, linewidths=0.3,
                           edgecolors='k', alpha=0.95)
    if edges_draw:
        nx.draw_networkx_edges(G, pos, edgelist=edges_draw,
                               width=widths, alpha=0.55, arrows=False)

    # vertical class labels (rotated)
    class_nodes = [n for n in nodes_with_pos if n.startswith("class:")]
    for n in class_nodes:
        x0, y0 = pos[n]
        label = G.nodes[n].get("label", "")
        plt.text(x0, y0 - 0.15, label, rotation=90,
                 ha='center', va='top', fontsize=7)

    # layer headers (left margin)
    for i, name in enumerate(["layer0","layer1","layer2","layer3","layer4","FC","class"]):
        plt.text(-0.03, -i, name, ha='right', va='center', fontsize=11, fontweight='bold')
    plt.xlim(-0.1, 1.05); plt.ylim(-6.6, 0.6)
    plt.title(title, fontsize=12); plt.axis('off')

plt.figure(figsize=(16, 10))
plt.subplot(1,2,1); draw_vertical(Gx, "Image circuit (x)")
plt.subplot(1,2,2); draw_vertical(Ge, "Explanation circuit (e)")
plt.tight_layout(); plt.show()

# Optional sanity: print top-1s
px = F.softmax(logits_x, dim=1)[0]; pe = F.softmax(logits_e, dim=1)[0]
ix = int(px.argmax()); ie = int(pe.argmax())
print(f"Image top-1: {imagenet_labels[ix] if ix < len(imagenet_labels) else ix} ({px[ix].item():.2f})")
print(f"Expl.  top-1: {imagenet_labels[ie] if ie < len(imagenet_labels) else ie} ({pe[ie].item():.2f})")
